# Encoder-Only — Hyperparameter Sweep (Lorenz)

Uses W&B Sweeps to systematically evaluate combinations of regularisation
strengths, latent dimension, and architecture choice for the encoder-only
representation learning model.

## How to use

1. **Cell 2** – set `WANDB_PROJECT`, `WANDB_ENTITY`, and the `sweep_config` dict.
2. **Cell 3** – define the `train_one_run` function (edit base overrides here).
3. **Cell 4** – create the sweep and launch agents.

To run multiple agents in parallel (e.g. on SLURM), copy the printed
`wandb agent ...` command and submit as separate jobs.

## 1 — Imports

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import wandb
from hydra.utils import instantiate
from omegaconf import OmegaConf

from JacobianODE.jacobians.core import seed_everything
from JacobianODE.jacobians.data import make_trajectories, postprocess_data, create_dataloaders
from JacobianODE.jacobians.training import train_model
from JacobianODE.encoder_only.config import load_encoder_config

torch.set_float32_matmul_precision('high')
print(f'PyTorch {torch.__version__}  |  GPUs: {torch.cuda.device_count()}')

## 2 — Sweep Configuration

Adjust `sweep_config` to define the search space.
- `method`: `'grid'`, `'random'`, or `'bayes'`
- For a quick test, reduce `count` in Cell 4.

In [ ]:
WANDB_PROJECT = 'Lorenz__EncoderOnly'
WANDB_ENTITY  = os.environ.get('WANDB_ENTITY', None)  # or set explicitly: 'my-team'

sweep_config = {
    'method': 'grid',  # 'grid' | 'random' | 'bayes'
    'name': 'encoder-only-reg-sweep',
    'metric': {'name': 'val/same_state_loss', 'goal': 'minimize'},
    'parameters': {
        # --- Regularisation weights ---
        'fnn_weight': {
            'values': [0.0, 0.001, 0.01, 0.1]
        },
        'amplification_weight': {
            'values': [0.0, 0.001, 0.01]
        },
        'decov_weight': {
            'values': [0.0, 0.001]
        },
        # --- Architecture ---
        # Uncomment to sweep over latent dimension:
        # 'n_latent': {'values': [5, 10, 20]},
    },
}

# For a Bayesian sweep over continuous ranges, use:
# sweep_config = {
#     'method': 'bayes',
#     'metric': {'name': 'val/same_state_loss', 'goal': 'minimize'},
#     'parameters': {
#         'fnn_weight':          {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1.0},
#         'amplification_weight':{'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 1.0},
#         'decov_weight':        {'distribution': 'log_uniform_values', 'min': 1e-4, 'max': 0.1},
#     },
# }

## 3 — Define `train_one_run`

This function is called once per sweep trial.  
Edit `base_overrides` to control fixed hyperparameters shared across all runs.

In [ ]:
# Fixed overrides shared by every sweep run
BASE_OVERRIDES = [
    'model=transformer',
    '++model.encoder.n_latent=10',
    '++model.encoder.d_model=64',
    '++model.encoder.n_layers=3',
    '++model.use_same_state_decoder=true',
    '++model.use_next_state_decoder=false',
    '++model.next_state_burn_in=0',  # burn-in for next-step loss (partial obs)
    # Training
    '++training.batch_size=32',
    '++training.lightning.optimizer_kwargs.lr=1e-4',
    '++training.trainer_params.max_epochs=150',
    '++training.trainer_params.limit_train_batches=200',
    '++training.early_stopping.early_stopping_patience=8',
    # Data
    '++data.trajectory_params.num_ics=32',
    '++data.trajectory_params.n_periods=12',
    '++data.trajectory_params.pts_per_period=100',
    '++data.train_test_params.seq_length=50',
    '++data.postprocessing.obs_noise=0.05',
]


def train_one_run():
    """Single sweep trial — called automatically by wandb.agent."""
    with wandb.init() as run:
        sweep_params = dict(run.config)  # hyperparams chosen by the sweep

        # Build Hydra overrides from sweep parameters
        sweep_overrides = []
        if 'fnn_weight' in sweep_params:
            sweep_overrides.append(f"++training.lightning.fnn_weight={sweep_params['fnn_weight']}")
        if 'amplification_weight' in sweep_params:
            sweep_overrides.append(f"++training.lightning.amplification_weight={sweep_params['amplification_weight']}")
        if 'decov_weight' in sweep_params:
            sweep_overrides.append(f"++training.lightning.decov_weight={sweep_params['decov_weight']}")
        if 'n_latent' in sweep_params:
            sweep_overrides.append(f"++model.encoder.n_latent={sweep_params['n_latent']}")

        # Load config
        cfg = load_encoder_config(overrides=BASE_OVERRIDES + sweep_overrides)

        # Reproducibility
        seed_everything(cfg.data.flow.random_state)

        # Data
        eq, sol, dt = make_trajectories(cfg)
        result = postprocess_data(cfg, sol['values'])
        values = result.values
        cfg.data.postprocessing.noise_scale_factor = result.noise_scale_factor
        cfg.data.postprocessing.mu = result.mu
        cfg.data.postprocessing.sigma = result.sigma

        train_dl, val_dl, _, trajs = create_dataloaders(cfg, values)
        n_obs = trajs['train_trajs'].sequence.shape[-1]

        # Model
        seed_everything(cfg.data.flow.random_state + 1)
        encoder  = instantiate(cfg.model.encoder, n_input=n_obs)
        lit_model = instantiate(
            cfg.training.lightning,
            encoder=encoder,
            n_obs=n_obs,
            context_margin=int(cfg.model.get('context_margin', 0)),
            next_state_burn_in=int(cfg.model.get('next_state_burn_in', 0)),
            use_same_state_decoder=bool(cfg.model.get('use_same_state_decoder', True)),
            use_next_state_decoder=bool(cfg.model.get('use_next_state_decoder', False)),
            decoder_hidden_dim=int(cfg.model.get('decoder_hidden_dim', 128)),
            decoder_n_layers=int(cfg.model.get('decoder_n_layers', 2)),
        )

        # Run name = sweep run id (W&B sets this automatically)
        name    = run.name
        project = WANDB_PROJECT

        # Train — uses the already-initialised wandb run
        train_model(cfg, lit_model, train_dl, val_dl, name=name, project=project, entity=WANDB_ENTITY)

## 4 — Create Sweep and Run Agent

`count` controls how many runs this agent executes.  
Set `count=None` for unlimited (agent stops when sweep is finished).

In [ ]:
# Create the sweep (only needs to be done once; subsequent cells can reuse sweep_id)
sweep_id = wandb.sweep(
    sweep_config,
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
)
print(f'Sweep ID: {sweep_id}')
print(f'Dashboard: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}/sweeps/{sweep_id}')

In [ ]:
# Run the sweep agent (blocks until `count` runs are complete)
# To resume an existing sweep, replace `sweep_id` with the existing ID string.
wandb.agent(sweep_id, function=train_one_run, count=24)

## 5 — Analyse Sweep Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

api = wandb.Api()
entity_str = WANDB_ENTITY or api.default_entity
sweep_runs = api.sweep(f'{entity_str}/{WANDB_PROJECT}/{sweep_id}').runs

records = []
for r in sweep_runs:
    if r.state == 'finished':
        rec = dict(r.config)
        rec['val_same_loss']    = r.summary.get('val/same_state_loss', float('nan'))
        rec['val_next_loss']    = r.summary.get('val/next_state_loss', float('nan'))
        rec['val_fnn_loss']     = r.summary.get('val/fnn_loss', float('nan'))
        rec['val_amp_loss']     = r.summary.get('val/amplification_loss', float('nan'))
        rec['latent_util']      = r.summary.get('val/latent_utilization', float('nan'))
        rec['run_name']         = r.name
        records.append(rec)

df = pd.DataFrame(records)
df = df.sort_values('val_same_loss')
print(f'{len(df)} finished runs')
df.head(10)

In [ ]:
# ---- Scatter: fnn_weight vs val_same_state_loss, coloured by amplification_weight ----
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(
    df['fnn_weight'],
    df['val_same_loss'],
    c=df['amplification_weight'],
    cmap='viridis',
    s=60, alpha=0.8, edgecolors='k', linewidths=0.4,
)
plt.colorbar(sc, ax=ax, label='amplification_weight')
ax.set_xscale('symlog', linthresh=1e-4)
ax.set_xlabel('fnn_weight')
ax.set_ylabel('val/same_state_loss')
ax.set_title('Sweep: regularisation vs. reconstruction loss')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Latent utilisation distribution ----
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df['latent_util'].dropna(), bins=20, color='steelblue', edgecolor='white')
ax.set_xlabel('latent utilisation (entropy-based, 0–1)')
ax.set_ylabel('count')
ax.set_title('Latent utilisation across sweep runs')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Best run summary ----
best = df.iloc[0]
print('Best run:')
for col in df.columns:
    print(f'  {col:30s}: {best[col]}')